<a href="https://colab.research.google.com/github/thomsmbockchildGod/assignment13-Intro-to-AI/blob/main/Assignment13AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
! pip install tensorflow

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

In [7]:
#Implement a basic text generation model using Python libraries such as TensorFlow (Keras, or PyTorch).

corpus = [
    "the cat is sleeping",
    "the dog is barking",
    "the bird is flying",
    "the cat is eating",
    "the dog is running"
]

#convert tokens into unique ID

tokenizer=Tokenizer()

tokenizer.fit_on_texts(corpus)

total_words=len(tokenizer.word_index)+1

print(total_words)

11


In [8]:

#Train the model on the selected dataset to generate text based on seed input.

#creating training sequence

input_sequences=[]

for line in corpus:
  token_list=tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    n_gram_sequence=token_list[:i+1]
    input_sequences.append(n_gram_sequence)

max_length=max([len(x) for x in input_sequences])

input_sequences=pad_sequences(input_sequences,maxlen=max_length,padding='pre')

print(input_sequences)

[[ 0  0  1  3]
 [ 0  1  3  2]
 [ 1  3  2  5]
 [ 0  0  1  4]
 [ 0  1  4  2]
 [ 1  4  2  6]
 [ 0  0  1  7]
 [ 0  1  7  2]
 [ 1  7  2  8]
 [ 0  0  1  3]
 [ 0  1  3  2]
 [ 1  3  2  9]
 [ 0  0  1  4]
 [ 0  1  4  2]
 [ 1  4  2 10]]


In [9]:
X=input_sequences[:,:-1]
Y=input_sequences[:,-1]
Y=tf.keras.utils.to_categorical(Y,num_classes=total_words)

In [10]:
model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_length-1))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(X,Y, epochs=200, verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1333 - loss: 2.3991
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.1333 - loss: 2.3921
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5333 - loss: 2.3851
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.3333 - loss: 2.3781
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.3333 - loss: 2.3707
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.3333 - loss: 2.3630
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.3333 - loss: 2.3548
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3333 - loss: 2.3459
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3333 - loss: 2.3362
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3333 - loss: 2.3255
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.3333 - loss: 2.3136
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.3333 - loss

In [12]:
#generating a function to predict next token

def generate_text(seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_length-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text


In [13]:
print(generate_text("the cat", next_words=5))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
the cat is eating sleeping flying is
